The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [2]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
# TODO(you): build `vocab` (the V most common words), `word2idx`, `idx2word`,
# and `corpus` (the token stream mapped to ids, dropping out-of-vocabulary words).
vocab = [x[0] for x in counts.most_common(V)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
corpus = [word2idx[w] for w in tokens if w in word2idx]


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [3]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        # TODO(you): embed the center ids and return the (B, V) scores over the whole
        # vocabulary (one score per possible context word). Cross-entropy + softmax are applied
        # by the loss in the training loop, so return the raw scores (logits), not probabilities.
        x = self.center(center_ids)
        return self.output(x)


In [4]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

# TODO(you): write the training loop. For each mini-batch, get the (B, V) logits from the center
# ids with model(...), compute the cross-entropy loss against the true context ids with loss_fn,
# backprop, and step the optimizer. Track the per-epoch loss. After training, set
# `emb = model.center.weight.detach().cpu().numpy()`.
data = torch.tensor(pairs, dtype=torch.long)
losses = []

for e in range(epochs):
    order = torch.randperm(len(data))
    total = 0

    for i in range(0, len(data), B):
        batch = data[order[i:i + B]]
        x = batch[:, 0]
        y = batch[:, 1]

        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()

        total += loss.item() * len(batch)

    avg = total / len(data)
    losses.append(avg)
    print("Epoch", e + 1, "loss", round(avg, 4))

emb = model.center.weight.detach().cpu().numpy()


Epoch 1 loss 6.9846
Epoch 2 loss 6.7125
Epoch 3 loss 6.5756


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [5]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

# TODO(you): compute `pca3` (N x 3) with PCA, and `umap3` with UMAP (guard UMAP in a
# try/except so a missing umap-learn does not crash the notebook).
pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap
    umap3 = umap.UMAP(n_components=3, random_state=0).fit_transform(X)
except ImportError:
    umap3 = None


/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [6]:
import plotly.graph_objects as go

# TODO(you): write `plot_embeddings(coords, words, query=None, neighbor_set=None)` that
# draws a plotly Scatter3d: hover text = the word; color/size the `query` and any words in
# `neighbor_set` distinctly. Return the figure (end the cell with the figure object).
def plot_embeddings(coords, words, query=None, neighbor_set=None):
    if neighbor_set is None:
        neighbor_set = set()

    colors = []
    sizes = []

    for w in words:
        if w == query:
            colors.append("red")
            sizes.append(9)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(7)
        else:
            colors.append("blue")
            sizes.append(3)

    fig = go.Figure(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        text=words,
        marker=dict(color=colors, size=sizes)
    ))

    return fig

plot_embeddings(pca3, plot_words)


## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [7]:
def neighbors(word, k=10):
    # TODO(you): return the k nearest words to `word` by cosine similarity over `emb`,
    # as a list of (word, score) sorted by descending score, excluding `word` itself.
    if word not in word2idx:
        return []

    x = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    i = word2idx[word]
    scores = x @ x[i]
    order = np.argsort(scores)[::-1]
    ans = []

    for j in order:
        if j != i:
            ans.append((idx2word[int(j)], float(scores[j])))
        if len(ans) == k:
            break

    return ans

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")


federal         0.822
troops          0.811
authorities     0.786
commonwealth    0.780
municipal       0.778
extending       0.777
occupying       0.772
invasion        0.772
contribution    0.771
marshal         0.768


In [8]:
# TODO(you): pick a query word, get its neighbors with neighbors(query, 10), and re-draw the
# projector with plot_embeddings(...) highlighting the query and its neighbors. A word only
# appears in the plot if it is among the N most frequent words used for pca3.
query = "government"
near = neighbors(query, 10)
near_words = {w for w, s in near}
coords = umap3 if umap3 is not None else pca3
plot_embeddings(coords, plot_words, query, near_words)


## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [9]:
words = ["government", "war", "said", "the"]

for word in words:
    print("\n" + word)
    for w, s in neighbors(word, 8):
        print(w, round(s, 3))



government
federal 0.822
troops 0.811
authorities 0.786
commonwealth 0.78
municipal 0.778
extending 0.777
occupying 0.772
invasion 0.772

war
outbreak 0.781
world 0.765
romania 0.764
transylvania 0.758
anglo 0.75
syrian 0.744
mysore 0.727
z 0.724

said
explained 0.897
felt 0.862
admitted 0.857
opined 0.854
told 0.853
thinks 0.838
realized 0.828
unsure 0.825

the
croatia 0.739
cuautla 0.734
montenegro 0.731
antiquaries 0.715
masonry 0.713
schooling 0.712
gallia 0.71
clerks 0.709


### My answers

Government and said were the two terms that gave me the most precise results. Government had terms like federal, authorities, and municipal. Said had terms like explained, admitted, and told. While war term had some related words, there were also some non-related words in this group. The term the got the worst results because it occurs near almost every other word.

It is possible that rare words have more likely bad neighbors because the model encounters them infrequently and therefore their vectors get updated less often.

In the 3D scatter plot, I could notice several clusters of political terms, countries and places, numerals and time terms, and common grammatical words. Although the related words were located near each other, the clusters were somewhat mixed. This could be due to the partial training of the model on the data and reduction of the dimensionality from 64 to just 3.